# Attention

Companion code for the *Predictive Analytics Lab* (section 22). Run the cell below.

In [ ]:
# Scaled dot-product self-attention, causal masking, and permutation equivariance
import numpy as np

rng = np.random.default_rng(42)
X = rng.normal(size=(4, 4))                         # 4 tokens, D = 4
Wq, Wk, Wv = (rng.normal(0, .7, (4, 3)) for _ in range(3))

def self_attention(X, causal=False):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    E = Q @ K.T / np.sqrt(K.shape[1])               # similarities, scaled by sqrt(D)
    if causal:
        E = np.where(np.triu(np.ones_like(E), 1) == 1, -np.inf, E)
    A = np.exp(E - E.max(axis=1, keepdims=True)); A /= A.sum(axis=1, keepdims=True)
    return A @ V, A

Y, A = self_attention(X); print("Attention weights:\n", A.round(2))
_, Am = self_attention(X, causal=True); print("Masked attention weights:\n", Am.round(2))
perm = [2, 0, 3, 1]
Yp, _ = self_attention(X[perm])
print("Permutation equivariant:", np.allclose(Yp, Y[perm]))
N, H = 100_000, 64
print(f"H x N x N attention weights in 16-bit floats: {H * N * N * 2 / 2 ** 30:,.0f} GB (the lecture's 1.192 TB)")